In [1]:
from pathlib import Path

import holidays
import pandas as pd

#### 1. Merge the separate raw datafiles

In [2]:
def read_all_files_in_dir(dir_paths):
    df = pd.DataFrame()

    for dir in dir_paths:
        # get all file names in the dir
        raw_files = [f for f in Path(dir).iterdir() if f.is_file()]

        # read and merge the files
        for file in raw_files: 
            curr_data = pd.read_csv(file)
            df = pd.concat([df, curr_data])

    return df

In [3]:
# combine all the raw files
raw_data_dir_paths = ['./data/raw/2020', './data/raw/2021']
full_dataset_df = read_all_files_in_dir(raw_data_dir_paths)

# sort by increasing start and end dates
full_dataset_df = full_dataset_df.sort_values(['starttime', 'stoptime'])

print('Total number of rows:', len(full_dataset_df))

Total number of rows: 393312


### 2. EDA

In [4]:
# check for missing values - None
def print_missing_values(df):
    num_rows_with_missing_values = df.isnull().any(axis=1).sum()
    print(df.isna().sum())
    print('Total rows:', len(df))
    print('Rows with missing values:', num_rows_with_missing_values)
    print('Proportion: {} %'.format(round(num_rows_with_missing_values/len(df) *100, 4)))

print_missing_values(full_dataset_df)

tripduration               0
starttime                  0
stoptime                   0
start station id           0
start station name         0
start station latitude     0
start station longitude    0
end station id             0
end station name           0
end station latitude       0
end station longitude      0
bikeid                     0
usertype                   0
birth year                 0
gender                     0
dtype: int64
Total rows: 393312
Rows with missing values: 0
Proportion: 0.0 %


### 2. Preprocessing the dataset

In [5]:
'''
Functions to process datetime.
Note: 
- no missing dates, so not handled 
- string values are returned, so that they are categorical values
- is_public_holiday is a boolean value
'''
us_holidays = holidays.US()

def convert_date_to_month_of_year(date):
    return str(date.month)

def convert_date_to_week_of_year(date):
    return str(date.isocalendar().week)

def convert_date_to_day_of_week(date):
    return str(date.dayofweek)

def convert_time_to_hour_of_day(time):
    return str(time.hour)

def convert_date_to_is_public_holiday(date):
    return 1 if date in us_holidays else 0

In [6]:
# force starttime into datetime format
full_dataset_df['starttime'] = pd.to_datetime(full_dataset_df['starttime'])

# create new date features
full_dataset_df['start_month_of_year'] = full_dataset_df['starttime'].apply(convert_date_to_month_of_year)
full_dataset_df['start_week_of_year'] = full_dataset_df['starttime'].apply(convert_date_to_week_of_year)
full_dataset_df['start_day_of_week'] = full_dataset_df['starttime'].apply(convert_date_to_day_of_week)
full_dataset_df['start_hour_of_day'] = full_dataset_df['starttime'].apply(convert_time_to_hour_of_day)
full_dataset_df['start_is_public_holiday'] = full_dataset_df['starttime'].apply(convert_date_to_is_public_holiday)

In [7]:
# drop columns that will not be used
columns_to_drop = [
    'tripduration',
    'starttime',
    'stoptime',
    'start station id',
    # 'start station name',
    # 'start station latitude',
    # 'start station longitude',
    'end station id',
    'end station name',
    'end station latitude',
    'end station longitude',
    'bikeid',
    'usertype',
    'birth year',
    'gender'
]
full_dataset_df = full_dataset_df.drop(columns=columns_to_drop)

In [8]:
full_dataset_df

,start station name,start station latitude,start station longitude,start_month_of_year,start_week_of_year,start_day_of_week,start_hour_of_day,start_is_public_holiday
0,Grove St PATH,40.719586,-74.043117,1,1,2,0,1
1,Grove St PATH,40.719586,-74.043117,1,1,2,0,1
2,Grove St PATH,40.719586,-74.043117,1,1,2,0,1
3,Sip Ave,40.730897,-74.063913,1,1,2,0,1
4,Grove St PATH,40.719586,-74.043117,1,1,2,0,1
...,...,...,...,...,...,...,...,...
23313,Newport PATH,40.727224,-74.033759,4,17,4,23,0
23314,Glenwood Ave,40.727551,-74.071061,4,17,4,23,0
23315,Jersey & 3rd,40.723332,-74.045953,4,17,4,23,0
23316,Marin Light Rail,40.714584,-74.042817,4,17,4,23,0
